In [1]:
import os
from dotenv import load_dotenv

load_dotenv() 

if os.getenv("LANGSMITH_API_KEY"):
    print("LangSmith API Key loaded")
if os.getenv("GOOGLE_API_KEY"):
    print("Google API Key loaded")

LangSmith API Key loaded
Google API Key loaded


In [2]:
import json
from langchain_core.documents import Document

def load_corpus(file_path: str) -> list[Document]:
    with open(file_path, "r") as f:
        corpus = json.load(f)
    
    return [
        Document(
            page_content=item["content"],
            metadata={"pmid": item["pmid"], "url": item["url"]}
        )
        for item in corpus
    ]

documents = load_corpus("./corpus.json")

In [3]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import EnsembleRetriever
from langchain_google_genai import ChatGoogleGenerativeAI

def build_retriever(documents: list[Document], k: int = 5) -> EnsembleRetriever:
    bm25_retriever = BM25Retriever.from_documents(documents, k=k)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(documents, embeddings)
    semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    return EnsembleRetriever(retrievers=[bm25_retriever, semantic_retriever], weights=[0.5, 0.5])

retriever = build_retriever(documents)
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """Answer the question based only on the context provided:
{context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)), 
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [5]:
response = chain.invoke("Çölyak hastalığı tanı kriterleri nelerdir?")

In [6]:
response

'Verilen metinlerde çölyak hastalığının tanı kriterlerine dair doğrudan bir liste bulunmamaktadır. Ancak, metinlerde çölyak hastalığı ile ilgili şu bilgiler yer almaktadır:\n\n*   **Tanı yöntemi:** Çölyak hastalığı, villöz atrofi (VA) ile karakterizedir ve tanısı için duodenal histoloji (biyopsi) gereklidir.\n*   **Endoskopik teknikler:** Villöz atrofinin belirlenmesi için standart beyaz ışıklı endoskopi (WLE) ve su daldırma tekniği, dar bant görüntüleme (NBI), boya bazlı kromoendoskopi, beyaz ışıklı büyütmeli endoskopi, i-scan ve konfokal endomikroskopi gibi gelişmiş endoskopik teknikler kullanılabilmektedir.\n*   **Yapay zeka kullanımı:** NHS histopatoloji laboratuvarında, duodenal biyopsilerin triyajında (sınıflandırılmasında) yapay zeka kullanımı, çölyak hastalığı tanısı alan vakaların raporlanma süresini kısaltmıştır.'

In [7]:
chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

'Based on the provided text, there is no information regarding antibiotic resistance patterns in community-acquired pneumonia.'

# Multi Query

In [8]:
# https://github.com/langchain-ai/rag-from-scratch/blob/main/rag_from_scratch_5_to_9.ipynb

multi_query_template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Return only the questions directly. Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(multi_query_template)

In [9]:
multi_query_chain = (
    {"question": RunnablePassthrough()}
    | prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [10]:
multi_query_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

['What are the current trends in antibiotic resistance for community-acquired pneumonia?',
 'How does antibiotic susceptibility vary in patients diagnosed with community-acquired pneumonia?',
 'What is the prevalence of drug-resistant pathogens in community-acquired pneumonia cases?',
 'Which antibiotics are becoming less effective against community-acquired pneumonia due to resistance?',
 'What are the clinical patterns of antimicrobial resistance observed in community-acquired pneumonia infections?']

In [11]:
multi_query_retrieval_chain = multi_query_chain | retriever.map()

In [12]:
multi_query_all_docs = multi_query_retrieval_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

In [13]:
len(multi_query_all_docs), len(multi_query_all_docs[0]), len(multi_query_all_docs[1]) 

(5, 7, 8)

In [14]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]
mq_chain = multi_query_retrieval_chain | get_unique_union | (lambda docs: "\n\n".join(d.page_content for d in docs))

In [15]:
all_docs = mq_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

/tmp/ipykernel_54496/3101763892.py:6: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


In [16]:
final_mq_template = """Answer the following question based on this context:

{context}

Question: {question}
"""

final_mq_prompt = ChatPromptTemplate.from_template(final_mq_template)

final_mq_chain = (
    {"context": mq_chain, 
     "question": RunnablePassthrough()} 
    | final_mq_prompt
    | llm
    | StrOutputParser()
)

final_mq_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

'Based on the provided context, there is no specific information detailing the antibiotic resistance patterns for community-acquired pneumonia (CAP).\n\nWhile the documents mention antibiotic resistance in other contexts—such as the multidrug-resistant nature of *Bordetella hinzii* (which is an emerging pathogen in patients with chronic lung disease), the general challenge of antibiotic resistance compromising treatment efficacy, and the role of antibiotic resistance in the over-prescribing of antibiotics for otitis media—they do not provide data on the specific resistance patterns of the pathogens that cause community-acquired pneumonia.'

# RAG Fusion

In [21]:
multi_query_all_docs[0][0].page_content

'External validation of risk scores and multivariate models for the diagnosis of community-acquired pneumonia in outpatients.While several risk scores for the diagnosis of community-acquired pneumonia (CAP) have been developed, they require prospective external validation. To externally validate existing prediction models, risk scores, and heuristics for the diagnosis of CAP in adults. The Enhancing Antibiotic Stewardship in Primary Care (EAST-PC) study recorded signs, symptoms, demographics, and vitals in 718 adults presenting to primary or urgent care clinics with acute lower respiratory tract infection between 2019 and 2023. C-reactive protein (CRP) was available for 575. The diagnosis of CAP was based on the clinician diagnosis and/or chest radiograph. Literature was searched for previous risk scores. Using the EAST-PC population, the area under the receiver operating characteristic curve (AUROCC), calibration curves, and percentage with CAP in each risk group were calculated for e

In [22]:
from langchain_core.documents import Document

def rrf(results: list[list[Document]], k: int = 60):
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = doc.page_content
            
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            
            fused_scores[doc_str] += 1 / (k + rank)

    reranked_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

    return [Document(page_content=content) for content, score in reranked_results]

In [24]:
from langchain_core.runnables import RunnableLambda
rag_fusion_chain = (
    multi_query_retrieval_chain
    | RunnableLambda(rrf)
)

In [25]:
rag_fusion_all_docs = rag_fusion_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

In [26]:
len(rag_fusion_all_docs)

13

In [27]:
final_rag_fusion_template = """Answer the following question based on this context:

{context}

Question: {question}
"""

final_rag_fusion_prompt = ChatPromptTemplate.from_template(final_rag_fusion_template)

final_rag_fusion_chain = (
    {"context": rag_fusion_chain | (lambda docs: "\n\n".join(d.page_content for d in docs)), 
     "question": RunnablePassthrough()} 
    | final_rag_fusion_prompt
    | llm
    | StrOutputParser()
)

final_rag_fusion_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

'Based on the provided context, there is limited information regarding specific antibiotic resistance patterns for community-acquired pneumonia (CAP). However, the following points are noted:\n\n*   **Multidrug-resistant pathogens:** The text identifies *Bordetella hinzii* as an emerging, under-recognized pathogen in patients with chronic lung disease that can be multidrug-resistant (the specific case mentioned was susceptible only to meropenem).\n*   **General concern regarding resistance:** The review on biomaterials for bacterial recruitment notes that the emergence of antibiotic resistance has substantially compromised the efficacy of conventional antibiotic-based therapeutic approaches for infectious diseases, including community-acquired infections.\n*   **Diagnostic and treatment implications:** The study comparing *Chlamydia psittaci* and *Legionella pneumophila* emphasizes that because these intracellular pathogens may remain undiagnosed without next-generation sequencing (NGS

# Decomposition

In [38]:
decomposition_template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output 3 queries. Return only the queries, no extra sentences."""

decomposition_prompt = ChatPromptTemplate.from_template(decomposition_template)


In [43]:
decomposition_query_chain = (
    {"question": RunnablePassthrough()}
    | decomposition_prompt
    | llm
    | StrOutputParser() | (lambda x: x.split("\n\n"))
)

In [47]:
questions = decomposition_query_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

In [48]:
questions

['What are the most common bacterial pathogens associated with community-acquired pneumonia and their current antibiotic resistance profiles?',
 'How do regional variations and local prescribing practices influence antibiotic resistance patterns in community-acquired pneumonia?',
 'What is the impact of prior antibiotic exposure on the development of resistant strains in patients diagnosed with community-acquired pneumonia?']

In [49]:
decomposition_multiple_template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(decomposition_multiple_template)

In [54]:
from operator import itemgetter

def format_q_a(question, answer):
    return f"Question: {question}\nAnswer: {answer}\n\n"
    

decomposition_chain = (
    {"question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs"),
     "context":  itemgetter("question") | retriever | (lambda docs: "\n\n".join(d.page_content for d in docs))
    }
    | decomposition_prompt
    | llm
    | StrOutputParser()
)

pairs = []
for question in questions:
    answer = decomposition_chain.invoke({"question": question, "q_a_pairs": "\n\n".join(pairs)})
    pairs.append(format_q_a(question, answer).strip())

print(q_a_pairs)



Question: What are the most common bacterial pathogens associated with community-acquired pneumonia and their current antibiotic resistance profiles?
Answer: Based on the provided documents, there is no comprehensive list of the most common bacterial pathogens associated with community-acquired pneumonia (CAP) or a summary of their general antibiotic resistance profiles.

However, the provided text highlights specific pathogens and clinical observations relevant to the topic:

*   **Atypical Pathogens:** *Chlamydia psittaci* and *Legionella pneumophila* are identified as common atypical pathogens that cause severe CAP. The study notes that both pathogens respond to **azithromycin** and that guideline-recommended **β-lactams/macrolide combination therapy** remains a standard approach, particularly when these intracellular pathogens are not specifically identified.
*   **Emerging/Under-recognized Pathogens:** *Bordetella hinzii* is described as an emerging, under-recognized Gram-negati

In [56]:
print(answer)

Based on the provided documents, there is no direct data or specific analysis quantifying the impact of prior antibiotic exposure on the development of resistant strains in patients diagnosed with community-acquired pneumonia (CAP).

However, the provided text offers the following relevant insights regarding the broader context of antibiotic use and resistance:

*   **General Drivers of Resistance:** The documents note that the high volume of antibiotic prescribing for respiratory tract infections (which account for 28.4% of prescriptions in Danish general practice) is a critical area for stewardship. Over-prescribing, often driven by diagnostic uncertainty in primary care, is explicitly identified as a factor that contributes to the development of antibiotic resistance.
*   **Clinical Complexity:** Because many CAP cases are treated empirically with broad-spectrum antibiotics (such as β-lactams/macrolide combinations) before specific pathogens are identified, the emergence of multidru

## Step Back

In [9]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

step_back_examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]

step_back_example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

step_back_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = step_back_example_prompt, examples=step_back_examples
)

step_back_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer.Here are a few examples:",
        ),
        step_back_few_shot_prompt,
        ("user", "{question}"),
    ]
)

step_back_query_chain = {"question": RunnablePassthrough()} | step_back_prompt | llm | StrOutputParser()

In [10]:
step_back_query_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

'What are the current trends and mechanisms of antibiotic resistance in common respiratory infections?'

In [11]:
from operator import itemgetter

step_back_response_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

step_back_response_prompt = ChatPromptTemplate.from_template(step_back_response_template)

step_back_final_chain = (
    {
        "normal_context": itemgetter("question") | retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
        "step_back_context": itemgetter("question") | step_back_query_chain | retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
        "question" : itemgetter("question")
    }
    | step_back_response_prompt
    | llm 
    | StrOutputParser()
)

In [12]:
step_back_final_chain.invoke({"question": "Antibiotic resistance patterns in community acquired pneumonia"})

'Based on the provided documents, there is no specific data detailing the prevalence or specific mechanisms of antibiotic resistance patterns in community-acquired pneumonia (CAP). However, the provided texts offer significant context regarding the clinical management, diagnostic challenges, and the broader implications of antibiotic use in CAP:\n\n*   **The Challenge of Undiagnosed Pathogens:** The literature emphasizes that atypical pathogens, such as *Chlamydia psittaci* and *Legionella pneumophila*, often remain undiagnosed without the use of next-generation sequencing (NGS). Because these pathogens can cause severe disease, the failure to identify them contributes to the reliance on broad-spectrum, guideline-recommended β-lactam/macrolide combination therapy.\n*   **Antibiotic Stewardship and Over-prescribing:** In Danish general practice, respiratory tract infections (including pneumonia) account for a significant portion of antibiotic prescriptions. A notable percentage of these

In [13]:
hyde_template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""

hyde_prompt = ChatPromptTemplate.from_template(hyde_template)

hyde_chain = (
    {"question": RunnablePassthrough()} | hyde_prompt | llm | StrOutputParser()
)

In [14]:
hyde_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

'**Title: Evolving Resistance Profiles in Community-Acquired Pneumonia: A Clinical Overview**\n\nThe landscape of community-acquired pneumonia (CAP) management is increasingly complicated by the shifting prevalence of antibiotic-resistant pathogens. Historically, *Streptococcus pneumoniae* remained reliably susceptible to beta-lactams and macrolides; however, recent surveillance data indicate a significant rise in penicillin-non-susceptible *S. pneumoniae* (PNSP) and multidrug-resistant (MDR) strains, often characterized by concurrent resistance to macrolides and tetracyclines. This trend is largely attributed to the widespread empirical use of broad-spectrum agents, which exerts selective pressure on the nasopharyngeal microbiome.\n\nFurthermore, the emergence of community-acquired methicillin-resistant *Staphylococcus aureus* (CA-MRSA) has necessitated a paradigm shift in empirical therapy, particularly in patients presenting with severe necrotizing pneumonia or post-influenza compli

In [18]:
hyde_response_template = """Answer the following question based on this context:

{context}

Question: {question}
"""

hyde_response_prompt = ChatPromptTemplate.from_template(hyde_response_template)

hyde_final_chain = (
    {
        "question": RunnablePassthrough(),
        "context": RunnablePassthrough() | hyde_chain | retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
    }
    | hyde_response_prompt 
    | llm 
    | StrOutputParser()
)

In [19]:
hyde_final_chain.invoke("Antibiotic resistance patterns in community acquired pneumonia")

'Based on the provided context, there is no specific information regarding the antibiotic resistance patterns of pathogens causing community-acquired pneumonia (CAP).\n\nWhile the text mentions that "the emergence of antibiotic resistance has substantially compromised treatment efficacy" in the context of general infectious diseases, and notes that *Chlamydia psittaci* and *Legionella pneumophila* respond to azithromycin, the provided documents do not detail specific resistance patterns for these or other CAP-related pathogens.'